In [1]:
import numpy as np

class DroneEKF:
    def __init__(self, beacon_pos, dt):
        # State: [x, y, z, roll, pitch, yaw]
        self.x = np.zeros((6, 1))
        self.P = np.eye(6) * 0.1  # Initial covariance
        
        # Process Noise (tune these based on IMU quality)
        self.Q = np.eye(6) * 0.01
        
        # Measurement Noise (variance of range sensors)
        self.R = np.eye(3) * 0.05
        
        self.beacon_pos = beacon_pos  # Shape (3, 3)
        self.dt = dt

    def predict(self, vel, gyro):
        """
        Prediction Step: Using IMU data
        vel: [vx, vy, vz] in body or global frame
        gyro: [p, q, r] (angular velocities)
        """
        # 1. State Transition (Simplified Kinematics)
        # Note: In a full drone model, you'd rotate body velocities to global
        self.x[0:3] += vel.reshape(3, 1) * self.dt
        self.x[3:6] += gyro.reshape(3, 1) * self.dt
        
        # 2. Jacobian of the transition function (F)
        # For this linear kinematic model, F is identity + small terms
        F = np.eye(6)
        
        # 3. Update Covariance
        self.P = F @ self.P @ F.T + self.Q

    def update(self, ranges):
        """
        Update Step: Using 3 Range Measurements
        ranges: [r1, r2, r3]
        """
        z = ranges.reshape(3, 1)
        
        # 1. Calculate Expected Ranges (h(x))
        h = np.zeros((3, 1))
        for i in range(3):
            dist = np.linalg.norm(self.x[0:3].flatten() - self.beacon_pos[i])
            h[i] = dist
            
        # 2. Calculate Jacobian (H) of the measurement function
        # H_i = [d(dist)/dx, d(dist)/dy, d(dist)/dz, 0, 0, 0]
        H = np.zeros((3, 6))
        for i in range(3):
            diff = self.x[0:3].flatten() - self.beacon_pos[i]
            dist = h[i, 0]
            if dist > 1e-6:
                H[i, 0:3] = diff / dist
        
        # 3. Kalman Gain
        S = H @ self.P @ H.T + self.R
        K = self.P @ H.T @ np.linalg.inv(S)
        
        # 4. Update State and Covariance
        y = z - h  # Innovation
        self.x = self.x + K @ y
        self.P = (np.eye(6) - K @ H) @ self.P

# --- Example Usage ---
beacons = np.array([
    [0, 0, 0],    # Beacon 1
    [10, 0, 0],   # Beacon 2
    [5, 10, 5]    # Beacon 3
])

ekf = DroneEKF(beacons, dt=0.01)

# Simulated step
velocity_imu = np.array([1.0, 0.2, 0.0]) # m/s
gyro_imu = np.array([0.0, 0.0, 0.1])     # rad/s
measured_ranges = np.array([5.1, 6.2, 8.5])

ekf.predict(velocity_imu, gyro_imu)
ekf.update(measured_ranges)

print(f"Estimated Position: {ekf.x[0:3].flatten()}")

Estimated Position: [3.61937032 1.69478244 0.58367465]
